# pbi-watchdog — teste no Fabric

Já configurado para as capacidades **capeusprodfabricf128001 (F128)** e
**capeusprodfabricf64001 (F64)**.

**Tudo roda em modo seguro:** `mode="observe"` e `dry_run=True`. Nada é cancelado, nada é
morto — é leitura pura. Nem exige permissão de escrita nos workspaces.

### Antes de rodar

**Anexe um Lakehouse** a este notebook (painel esquerdo → Add → Lakehouse) se quiser que a
baseline sobreviva entre execuções. Para um teste rápido, deixe `USAR_LAKEHOUSE = False`.

### Por que a descoberta por `sempy` falhava

O Capacity Metrics App vem do AppSource, que instala o conteúdo no **Meu workspace** de
quem instalou — a URL do relatório mostra isso: `app.powerbi.com/groups/me/apps/<id>/...`.
Como não existe um workspace para listar, nenhuma busca por workspaces o encontra. A
descoberta abaixo usa a REST API e varre **apps instalados**, Meu workspace e workspaces.

## 1. Instalação

`%pip` reinicia o interpretador — por isso fica sozinho, antes de qualquer variável.

Exige **pbi-watchdog 0.1.2 ou superior**: é a versão que sabe ler datasets de app instalado.

> Para agendar, registre a lib num Environment em vez de usar `%pip` a cada execução.
> E não instale `pbi-watchdog[fabric]`: o runtime do Fabric já traz o `sempy`.

In [ ]:
%pip install --upgrade pbi-watchdog

## 2. Descobrir o Capacity Metrics App

Varre apps instalados, Meu workspace e workspaces. Se o app estiver no Meu workspace, o
candidato vem sem `workspace_id` — e isso é o correto: o `executeQueries` usa a rota
`/datasets/{id}` nesse caso.

In [ ]:
from pbi_watchdog.auth import build_token_provider
from pbi_watchdog.config import NotebookAuth
from pbi_watchdog.discover import find_metrics_app_rest, render
from pbi_watchdog.rest import RestClient

client = RestClient(build_token_provider(NotebookAuth()))

candidatos = find_metrics_app_rest(client)
if not candidatos:
    print("Nada com nome óbvio; repetindo sem filtro de nome...\n")
    candidatos = find_metrics_app_rest(client, include_all=True)

print(render(candidatos))

METRICS = candidatos[0] if candidatos else None

Se vier mais de um candidato e o primeiro não for o certo, escolha na mão — por exemplo
`METRICS = candidatos[1]`. O nome do dataset do Metrics App costuma ser
`Fabric Capacity Metrics`.

Se vier **vazio mesmo com `include_all=True`**, o problema não é o nome: ou o app não está
instalado para esta identidade, ou o tenant não liberou *Dataset Execute Queries REST API*
(Admin Portal → Tenant settings → Integration settings).

## 3. Capacidades monitoradas

Fixadas nas duas que interessam. Os GUIDs vieram do `list_capacities()` da sua tenant —
a PPU (Premium Per User) fica de fora de propósito.

In [ ]:
CAPACIDADES = [
    {"key": "F128_PROD",    "id": "7a56cab6-83d1-476a-bb52-6c6a6bd9eeaa", "sku": "F128"},
    {"key": "F64_SANDBOX",  "id": "c87f7f9c-7c57-4707-8c1f-c945ac14f621", "sku": "F64"},
]

for c in CAPACIDADES:
    print(f"  {c['key']:<16} sku={c['sku']:<6} id={c['id']}")

# Para conferir a lista completa da tenant:
# import sempy.fabric as fabric
# display(fabric.list_capacities())

## 4. Montar a config

Nenhum GUID de dataset digitado à mão: o bloco `metrics_source` sai do candidato
descoberto na célula 2.

In [ ]:
from pbi_watchdog import WatchdogConfig

# True  = grava no Lakehouse anexado (a baseline sobrevive — use isto de verdade)
# False = SQLite temporário do driver; some ao fim da sessão, serve só para testar
USAR_LAKEHOUSE = True

FUSO = "America/Sao_Paulo"
TEAMS_WEBHOOK_URL = ""   # opcional neste teste

assert METRICS is not None, "Rode a célula 2: nenhum Metrics App identificado."

storage = (
    {"kind": "delta", "table_prefix": "watchdog_"}
    if USAR_LAKEHOUSE
    else {"kind": "sqlite", "path": "/tmp/watchdog_teste.db"}
)

config = WatchdogConfig.from_dict(
    {
        "version": 1,
        "timezone": FUSO,
        "auth": {"kind": "notebook"},
        # metrics_app_rest (e não sempy) porque o dataset está num app instalado:
        # o sempy só enxerga workspaces.
        "metrics_source": METRICS.as_config(kind="metrics_app_rest"),
        "storage": storage,
        "defaults": {
            "mode": "observe",         # nunca age; só detecta e registra
            "interval_minutes": 15,    # precisa bater com o agendamento, quando houver
            "baseline": {"lookback_days": 7, "min_days": 4, "method": "median"},
            "thresholds": {"alert": 1.2, "throttle": 1.5, "kill": 1.8},
            "guards": {
                "min_cu_seconds": 300,
                "consecutive_breaches": 2,
                "cooldown_minutes": 60,
                "max_actions_per_run": 5,
            },
            "protect": {"name_patterns": ["(?i)regulat", "(?i)\\bantt\\b", "(?i)executiv"]},
            "notify": ([{"kind": "teams", "url": TEAMS_WEBHOOK_URL}] if TEAMS_WEBHOOK_URL else []),
        },
        "capacities": CAPACIDADES,
    }
)

print("Metrics App :", METRICS.workspace_name, "/", METRICS.dataset_name)
print("dataset_id  :", METRICS.dataset_id)
print("workspace_id:", METRICS.workspace_id or "(omitido — Meu workspace)")
print("storage     :", storage["kind"])
for c in config.enabled_capacities:
    print(f"  {c.key:<16} modo={c.policy.mode} sku={c.sku}")

## 5. Diagnóstico

Cada checagem testa algo concreto e, quando falha, diz o que resolve.

A que mais importa aqui é **Perfil de DAX**: o modelo do Metrics App muda de nome entre
versões, e é o ponto onde a integração costuma quebrar. Se ela falhar, vá para a célula 6.

In [ ]:
from pbi_watchdog.doctor import Doctor, render as render_doctor

print(render_doctor(Doctor(config).run(deep=True)))

## 6. Só se o perfil de DAX falhar

Mostra as tabelas reais do seu Metrics App e o que falta para cada perfil conhecido.

Com isso dá para preencher `metrics_source.dax_override`: qualquer DAX serve, desde que
devolva `item_id, item_name, item_kind, workspace_id, workspace_name, cu_seconds_today`,
usando `{capacity_id}` como placeholder.

In [ ]:
from pbi_watchdog.sources import MetricsAppRestSource
from pbi_watchdog.sources.profiles import (
    PROBE_ORDER,
    PROFILES,
    join_tables_and_columns,
    missing_requirements,
    table_names,
)

source = MetricsAppRestSource(config.metrics_source, client)
raw_tabelas, raw_colunas = source._introspect()
tabelas = table_names(raw_tabelas)
colunas = join_tables_and_columns(raw_tabelas, raw_colunas)

print(f"== {len(tabelas)} tabelas no modelo ==")
for t in sorted(tabelas):
    print("  ", t)

print("\n== Compatibilidade dos perfis ==")
for nome in PROBE_ORDER:
    faltando = missing_requirements(PROFILES[nome], tabelas, colunas)
    print(f"  {'❌' if faltando else '✅'} {nome}" + (f" — falta: {', '.join(faltando)}" if faltando else ""))

# Para ver as colunas de uma tabela específica:
# print([c for c in colunas if c.startswith("Items[")])

## 7. Primeiro ciclo

O primeiro ciclo é sempre **bootstrap**: tira um snapshot do CU acumulado do dia e não tem
com o que comparar. Isso é esperado, não é erro.

O consumo de um intervalo só aparece no **segundo** ciclo, e a classificação só começa
depois de `min_days` (4) dias de histórico no mesmo bucket horário.

In [ ]:
from pbi_watchdog import Watchdog

wd = Watchdog(config, dry_run=True)
try:
    for s in wd.run_once():
        print(f"[{s.capacity_key}] modo={s.mode} | {s.items_scanned} itens | "
              f"{s.anomalies} anomalias | {s.actions_taken} ações")
        if s.extra.get("bootstrap"):
            print("   (primeiro snapshot — rode de novo daqui a ~15 min)")
        for e in s.errors:
            print("   ! erro:", e)
        for ev in s.events:
            print(f"   {ev.tier} → {ev.effective_tier} | {ev.item_name} | "
                  f"ratio={ev.ratio:.2f} | cu={ev.cu_seconds:,.0f} | "
                  f"contido por: {ev.suppressions or '—'}")
finally:
    wd.close()

## 8. Ver a mecânica funcionando agora

Rode a célula 7 de novo em ~15 minutos para ver o consumo do intervalo. Mas a
classificação depende de dias de histórico — então a célula abaixo demonstra a detecção
com dados **sintéticos**, sem tocar na sua tenant.

Repare no resultado: mesmo um pico de 10x fica em `alert`, contido por `observe_mode` e
`awaiting_consecutive_breaches`. É exatamente o comportamento que se quer no começo.

In [ ]:
import datetime as dt

from pbi_watchdog.core import baseline as bl, detect
from pbi_watchdog.models import Interval

agora = dt.datetime(2026, 7, 22, 14, 15)
policy = config.enabled_capacities[0].policy


def intervalo(cu, quando, minutos=15):
    return Interval(
        capacity_key="DEMO", item_id="item-x", item_name="Relatório de Vendas",
        item_kind="SemanticModel", workspace_id="ws-1", workspace_name="WS",
        window_start=quando, window_end=quando + dt.timedelta(minutes=minutos), cu_seconds=cu,
    )


historico = [intervalo(1000.0, dt.datetime(2026, 7, 15 + d, 14, 0)) for d in range(7)]
baselines = bl.compute_baselines(
    historico, policy.baseline, target_bucket="h14",
    reference_date=agora.date(), target_minutes=policy.interval_minutes, max_stretch=3.0,
)
base = baselines["item-x"]
print(f"Baseline: {base.value:,.0f} CU·s  ({base.days} dias, {base.samples} amostras, {base.method})\n")

for rotulo, cu in [("normal", 1050), ("alerta", 1300), ("throttle", 1700), ("pico", 10000)]:
    a = detect.assess_one(intervalo(cu, agora), base, policy, None, now=agora)
    print(f"  {cu:>6,} CU·s ({rotulo:<8}) → ratio {(a.ratio or 0):>5.2f} | "
          f"detectado={a.tier.label:<9} aplicado={a.effective_tier.label:<9} "
          f"| {', '.join(a.suppressions) or 'sem travas'}")

## 9. Calibração — depois de 1 a 2 semanas

Faz replay do histórico e responde: *com estes limiares, quantas vezes eu teria matado
alguma coisa, e o quê?* É o que diz se os thresholds servem para o seu ambiente e quais
itens pertencem à lista de protegidos.

Só produz números úteis depois de vários dias em `observe`.

In [ ]:
from pbi_watchdog.calibrate import calibrate_all, render as render_calib

for report in calibrate_all(config, days=14):
    print(render_calib(report))
    print()

## 10. O histórico gravado

Com `USAR_LAKEHOUSE = True` são tabelas Delta comuns — dá para montar um relatório Power BI
de FinOps direto em cima delas.

`watchdog_events` guarda **também o que o watchdog decidiu não fazer**, na coluna
`suppressions`. É ela que permite calibrar: você vê o que quase aconteceu.

In [ ]:
if USAR_LAKEHOUSE:
    display(spark.sql("SELECT * FROM watchdog_snapshots ORDER BY ts DESC LIMIT 20"))
    # display(spark.sql("SELECT * FROM watchdog_intervals ORDER BY window_end DESC LIMIT 50"))
    # display(spark.sql("SELECT * FROM watchdog_events ORDER BY ts DESC LIMIT 50"))
else:
    import sqlite3

    import pandas as pd

    con = sqlite3.connect("/tmp/watchdog_teste.db")
    display(pd.read_sql("SELECT * FROM snapshots ORDER BY ts DESC LIMIT 20", con))
    con.close()

## Próximos passos

1. **Agende este notebook a cada 15 min** (bata com `interval_minutes`). No agendado, só as
   células 3, 4 e 7 são necessárias.
2. **Deixe rodar 2 a 4 semanas em `observe`.** Sem baseline, enforcement mata carga legítima.
3. **Rode a calibração** e ajuste `thresholds`, `min_cu_seconds` e `protect`.
4. Só então considere `mode="enforce"`, começando pela **F64_SANDBOX** e com
   `max_actions_per_run` baixo. A F128 de produção fica em `observe` por bem mais tempo.

Para agir (cancelar refresh/jobs), a identidade precisa ser **Member ou Admin** dos
workspaces monitorados. Só observar não exige permissão de escrita nenhuma.

Repositório: <https://github.com/devrenanferrari/Watchdog-powerbi>